In [ ]:
!pip install --upgrade langchain-google-genai
!pip install "langchain>=0.2.7" \ "langchain-core>=0.2.6" \ "langchain-community>=0.2.6"
!pip install -qU "langchain-chroma>=0.1.2"
!pip install cohere
!pip install langchain-cohere
!pip install --upgrade langchain langchain-community
!pip install anthropic
!pip install openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencie

In [ ]:
!pip install --upgrade google-genai

In [ ]:
import os
from google.colab import userdata




try:
    os.environ["GEMINI_API_KEY"] = userdata.get("Muwaffaq_key_gemni")
    os.environ["GOOGLE_API_KEY"] = userdata.get("Muwaffaq_key_gemni")
except userdata.SecretNotFoundError:
    print("ERROR: Please create a secret in Google Colab called 'GEMINI_API_KEY' and add your API key.")

# Nursultan's WAY!

In [ ]:
# @title Creating Dataset
import pandas as pd
import json

def process_research_data(csv_file_path, output_json_path=None):
    """
    Reads a CSV, filters for specific father_departments,
    extracts 60 random samples per department, and returns a JSON list.
    """

    # 1. Load the dataset
    # We assume the CSV is comma-separated. If it's different, change sep=','
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        return "Error: The file was not found."
    except Exception as e:
        return f"Error reading CSV: {e}"

    # 2. Define the target departments to filter
    target_departments = [
        "العبادات",
        "الزواج والطلاق والنفقات",
        "المعاملات والأمانات",
        "الحظر والإباحة واللباس والزينة",
        "الذكر والتزكية والأخلاق"
    ]

    # Filter the dataframe to only include these departments
    filtered_df = df[df['father_department'].isin(target_departments)]

    # 3. Get 60 random samples from each department
    # We use a lambda function to sample.
    # n=60 specifies the number of items.
    # replace=False ensures we don't get duplicates (unless the group is smaller than 60, handled below).

    def sample_group(group):
        # If the group has fewer than 60 rows, take all of them.
        # Otherwise, take 60 random rows.
        n = min(len(group), 60)
        return group.sample(n=n, random_state=42) # random_state ensures reproducibility

    sampled_df = filtered_df.groupby('father_department', group_keys=False).apply(sample_group)

    # 4. Prepare columns for output
    # The user requested specific keys: question, answer, title, id, son_department, _fatherdepartment

    # Rename columns to match the desired output keys
    # Assuming the CSV columns are: father_department, son_department, title, question, answer, Id
    sampled_df = sampled_df.rename(columns={
        'Id': 'id',
        'father_department': '_fatherdepartment'
        # son_department, title, question, answer usually map directly,
        # but ensure case matches if your CSV is different.
    })

    # Select only the requested columns in the specific order
    cols_to_keep = ['question', 'answer', 'title', 'id', 'son_department', '_fatherdepartment']

    # Check if all columns exist to avoid errors
    available_cols = [c for c in cols_to_keep if c in sampled_df.columns]
    final_df = sampled_df[available_cols]

    # 5. Convert to JSON
    # orient='records' creates a list of objects: [{...}, {...}]
    # force_ascii=False is crucial for rendering Arabic characters correctly
    json_output = final_df.to_json(orient='records', force_ascii=False, indent=4)

    # Optionally save to a file
    if output_json_path:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            f.write(json_output)
        print(f"Data successfully saved to {output_json_path}")

    return json_output

# --- Usage Example ---
if __name__ == "__main__":
    # Replace 'data.csv' with your actual file name
    # Ensure your CSV has columns: father_department, son_department, title, question, answer, Id
    csv_file = '/content/فتاوى جاهزة - فتاوى جاهزة.csv'

    # This will print the JSON to the console or save it if a path is provided in the function
    json_result = json.loads(process_research_data(csv_file, 'output_samples.json'))

    # Print first 500 characters to verify output
    print(json_result[:5])

Data successfully saved to output_samples.json
[{'question': 'ما حكم كلمة (باي)؟ هل هي حرام، أم غير مستحبة؟', 'answer': 'أقول وبالله التوفيق: البحث فيها متعلق بالأولية والأفضلية لا بالحرمة، فالأفضل أن يكون خطابنا بالعربية، والله أعلم.', 'title': 'حكم قول كلمة «باي»', 'id': 2661, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر والإباحة واللباس والزينة'}, {'question': 'ما حكم النظر إلى ظهر وبطن المرأة من خلال التلفاز؟', 'answer': 'أقول وبالله التوفيق: يجب التجنب لذلك بقدر الاستطاعة، والله أعلم.', 'title': 'رؤية العورة من خلال التلفاز', 'id': 5609, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر والإباحة واللباس والزينة'}, {'question': 'لي جاره نصرانيه طلبت مني من لحم الاضحيه اعتقد أنها تتبارك فيه، هل يجوز أن أطعمها من أضحيتي أم تقتصر على المسلمين؟', 'answer': 'أقول وبالله التوفيق: يجوز إطعام الأضحية لمن شاء من مسلمين أو غيرهم، والله أعلم.', 'title': 'إطعام النصارى من لحم الأضحية', 'id': 8284, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر 

/tmp/ipython-input-1919655724.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = filtered_df.groupby('father_department', group_keys=False).apply(sample_group)


In [ ]:
print(json_result[:5])
json_list = json_result

[{'question': 'ما حكم كلمة (باي)؟ هل هي حرام، أم غير مستحبة؟', 'answer': 'أقول وبالله التوفيق: البحث فيها متعلق بالأولية والأفضلية لا بالحرمة، فالأفضل أن يكون خطابنا بالعربية، والله أعلم.', 'title': 'حكم قول كلمة «باي»', 'id': 2661, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر والإباحة واللباس والزينة'}, {'question': 'ما حكم النظر إلى ظهر وبطن المرأة من خلال التلفاز؟', 'answer': 'أقول وبالله التوفيق: يجب التجنب لذلك بقدر الاستطاعة، والله أعلم.', 'title': 'رؤية العورة من خلال التلفاز', 'id': 5609, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر والإباحة واللباس والزينة'}, {'question': 'لي جاره نصرانيه طلبت مني من لحم الاضحيه اعتقد أنها تتبارك فيه، هل يجوز أن أطعمها من أضحيتي أم تقتصر على المسلمين؟', 'answer': 'أقول وبالله التوفيق: يجوز إطعام الأضحية لمن شاء من مسلمين أو غيرهم، والله أعلم.', 'title': 'إطعام النصارى من لحم الأضحية', 'id': 8284, 'son_department': 'الحظر والإباحة', '_fatherdepartment': 'الحظر والإباحة واللباس والزينة'}, {'question': 'ما حك

In [ ]:
system_instructions= """
**System Persona:** You are an expert Islamic scholar specializing in Fiqh, specifically adhering to the **Hanafi Madhab**.
**Objective:** Answer user inquiries using the Arabic language (`al-Fusha` or accessible standard Arabic).

**Strict Constraints:**
* **Source Material:** You must strictly cite rulings based on the Hanafi school. Do not offer opinions from other schools (Maliki, Shafi'i, or Hanbali) unless explicitly asked for a comparison.
* **Tone:** Respectful, authoritative, and concise. Avoid unnecessary filler words.
* **Language:** All responses must be in Arabic.
* *concise*: make the answer consice and to the point

**Task:** Analyze the user's question and provide a direct Hanafi ruling.

"""

In [ ]:
# @title Gemnin
from google import genai
from google.genai import types
def read_gemnin(query):
   print("STARTING GEMINI")
   client = genai.Client()
   config = types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(
                    thinking_budget=-1,
                    include_thoughts=False),
                system_instruction=system_instructions,
            )
   contents =[types.Content(role="user", parts=[types.Part(text=f" user question is :{query}")])]

   model = "gemini-2.5-flash"
   response = client.models.generate_content(model=model,
                                                      contents=contents,
                                                      config=config)
   print(response.text)
   return response.text

In [ ]:
from openai import OpenAI
def read_gpt(query):
  print("STARTING GPT")
  client = OpenAI(api_key="YOUR_OPENAI_API_KEY")
  response = client.responses.create(
      model="gpt-5.1",
      reasoning={"effort": "low"},
      instructions=system_instructions,
      input=f"user question is :{query}",
  )

  print(response.output_text)
  return response.output_text

In [ ]:
import time
import random

curated_dataset = []
failed_records = []

# Configuration
MAX_RETRIES = 3
BASE_DELAY = 2 # Seconds to wait on success
ERROR_DELAY = 5 # Initial seconds to wait on error

print(f"Starting processing of {len(json_list[254:])} items...")

for index, j in enumerate(json_list[253:254]):
    query = j.get("question")

    # Retry mechanism
    retries = 0
    success = False

    while retries < MAX_RETRIES and not success:
        try:
            print(f"Processing item {index + 1}/{len(json_list[254:])}: {query[:30]}...")

            # --- Your Core Logic ---
            llm_answer = read_gpt(query)
            # -----------------------

            # Append successful result
            curated_dataset.append({
                **j,
                "llm_answer": llm_answer,
                "llm": "gpt-5.1",
                "is_thinking": True
            })
             # --- Your Core Logic ---
            llm_answer = read_gemnin(query)
            # -----------------------

            # Append successful result
            curated_dataset.append({
                **j,
                "llm_answer": llm_answer,
                "llm": "gemini-2.5-flash",
                "is_thinking": True
            })

            success = True

            # Rate Limit Protection: Sleep briefly on success to be polite to the API
            # Adding a small random jitter helps prevent synchronized API hits
            time.sleep(BASE_DELAY + random.uniform(0, 1))

        except Exception as e:
            retries += 1
            wait_time = ERROR_DELAY * retries # Linear backoff (5s, 10s, 15s)

            print(f"⚠️ Error on item {index} (Attempt {retries}/{MAX_RETRIES}): {e}")

            if retries < MAX_RETRIES:
                print(f"   Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed to process item {index} after {MAX_RETRIES} attempts.")
                # Log the failure so you don't lose track of it
                failed_records.append({"item": j, "error": str(e)})

# Summary
print("--- Processing Complete ---")
print(f"Successfully curated: {len(curated_dataset)}")
print(f"Failed items: {len(failed_records)}")

Starting processing of 46 items...
Processing item 1/46: شخص يريد العمل في شركة تقوم عل...
STARTING GPT


العمل الموصوف في أصلِه يدخل تحت باب **السمسرة / الوكالة بأجرة (الجعالة)**، وهي جائزة في المذهب الحنفي بشروط، فيجوز أخذ نسبة من الربح أو من ثمن السلعة مقابل الدلالة على المشتري أو التسويق للمنتج.

**الضوابط في المذهب الحنفي ليكون العمل مباحًا:**

1. **أن تكون السلع مباحة** في ذاتها واستعمالها (مثل الهواتف والأجهزة الكهربائية المباحة).
2. **أن تكون العمولة (النِّسبة)** معلومة متفقًا عليها مع الشركة، ولا يترتّب عليها جهالة فاحشة أو نزاع.
3. **ألا تُؤخذ العمولة من حقّ العاملين الآخرين**، بل تكون من حصة الشركة أو من اتفاق مستقل معها؛ فإذا كان كل واحد يأخذ عمولته على حسب جهده، ولا يُنقص من الآخرين، فهذا جائز.
4. **ألا يكون النظام هرمِيًّا محرّمًا** يقوم أساسًا على:
   - أخذ أموال من المشتركين الجدد (رسوم اشتراك كبيرة) مقابل وعود أرباح مستقبلية،
   - أو أن يكون الغالب في الربح من ضمّ الأفراد لا من بيع السلعة نفسها؛  
   فهذا داخل في المعاملات الفاسدة عند الحنفية؛ لما فيه من الغرر وأكل المال بالباطل.

**النتيجة (على مذهب الحنفية):**

- إذا كان العمل مجرّد تسويق لمنتجات حلال، والعمولة على قدر ا

In [ ]:
print(len(curated_dataset))
curated_dataset.remove(curated_dataset[1])
print(len(curated_dataset))

2
1


In [ ]:
import json

file_path_1 = "/content/results_final_2.json"
file_path_2 = "/content/results_final_254.json"

# Load the first JSON file
with open(file_path_1, 'r', encoding='utf-8') as f:
    list_1 = json.load(f)

# Load the second JSON file
with open(file_path_2, 'r', encoding='utf-8') as f:
    list_2 = json.load(f)

# Concatenate the two lists
concatenated_list = list_1 + list_2 + curated_dataset

print(f"Length of the first list: {len(list_1)}")
print(f"Length of the second list: {len(list_2)}")
print(f"Length of the concatenated list: {len(concatenated_list)}")

Length of the first list: 92
Length of the second list: 507
Length of the concatenated list: 600


In [ ]:
import json

file_path = "/content/results_final_llm_answers.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(concatenated_list, f, ensure_ascii=False, indent=2)

print(f"Curated dataset saved to {file_path}")

Curated dataset saved to /content/results_final_llm_answers.json


# LLM as a Judge Fudge!

In [ ]:
system_instruction= """
You are given two answers Answer A and
answer B both enclosed within triple backticks. Your task
is to evaluate how semantically close are the answers.
Answer A: {{{original_answer}}}
Answer B: {{{generated_answer}}}

**Your response MUST be a single number object. Do not include any text, explanations, or code outside of the number.**

- The value for the **"Correctness"** key must be one of the following strings: **"Yes"** or **"No"**. (This evaluates: Is Answer B correct given the reference answer A.)
- The value for the **"Rating"** key must be an integer: **1, 2, or 3**.
    - **-1:** They are contradictory or opposite (worng answer).
    - **0:** They agree and disagree in some aspects (partially correct).
    - **1:** They have identical meaning (correct).

"""

In [ ]:
import anthropic
from pydantic import BaseModel
from anthropic import Anthropic, transform_schema

class ContactInfo(BaseModel):
    correctness: str
    rating: int

def judge_claude(answerA, answerB):
  key = "YOUR_OPENAI_API_KEY"
  client = anthropic.Anthropic(api_key=key)

  response = client.messages.create(
      model="claude-sonnet-4-5",
      system=system_instruction,
      max_tokens=10000,
      thinking={
          "type": "enabled",
          "budget_tokens": 5000
      },
      messages=[{
          "role": "user",
          "content": [{
                    "type": "text",
                    "text": f"## Answer A is:```{answerA}```. \n ## Answer B is : ```{answerB}```"
                }]


      }],

  )
  answer = ""
  for block in response.content:
    if block.type == "thinking":
        print(f"\nThinking summary: {block.thinking}")
    elif block.type == "text":
        print(f"\nResponse: {block.text}")
        answer = block.text
  print("--"*50)
  return answer


print(judge_claude("i'm smart", "I'm clever"))




Thinking summary: Let me analyze the semantic similarity between Answer A and Answer B.

Answer A: "i'm smart"
Answer B: "I'm clever"

These two answers are expressing very similar ideas. Both "smart" and "clever" are synonyms that refer to intelligence or mental capability. The only differences are:
1. Capitalization (which is not semantically significant)
2. The choice between "smart" and "clever" (which are very close synonyms)

So let me evaluate:

**Correctness**: Is Answer B correct given the reference answer A?
- Answer A says the person is smart
- Answer B says the person is clever
- Smart and clever are synonyms, so Answer B conveys essentially the same meaning
- Therefore: **Yes**

**Rating**: How semantically close are they?
- -1: Contradictory or opposite - NO, they are not contradictory
- 0: Agree and disagree in some aspects - This could be argued, but "smart" and "clever" are very close synonyms
- 1: Identical meaning - This is the most appropriate. While the words are 

In [ ]:
print(concatenated_list[:5])

[{'question': 'سائق تاكسي وجد هاتفًا في سيارته منذ شهرين، ولم يصل لصاحبه إلى الآن، هل يجوز له استعماله؟', 'answer': 'أقول وبالله التوفيق: هذا الهاتف يكون لقطة، فإن كان السائق الملتقط له فقيراً يجوز أن يتصدق به على نفسه؛ وإن كان غنياً يتصدق به على غيره، والله أعلم.', 'title': 'التصرف باللقطة', 'id': 6129, 'son_department': 'اللقطة', '_fatherdepartment': 'المعاملات والأمانات', 'llm_answer': 'الهاتف في هذه الحالة يُعدُّ **لُقَطَة** في فقه الحنفية، وحكمه ما يلي:\n\n1. **لا يجوز له استعمال الهاتف** ما دام لم يأذن له صاحبه؛ لأن ملكيته ثابتة لمالكه المجهول، واستعماله يكون غصبًا وتصرفًا في ملك الغير بغير إذنه.  \n2. يجب عليه أن **يتعاهد حفظه**، وأن **يُعَرِّفَه** صاحبه بما جرت به العادة:\n   - سؤال الركّاب المعتادين إن كان يعرفهم.  \n   - مراجعة شركة الاتصال إن أمكن للوصول لصاحب الرقم.  \n   - وضع إعلان مناسب في المكان أو التطبيق الذي يُحجز به التاكسي (إن وُجد)، أو في الموقف المعتاد… إلخ، بما لا يفضح صاحب الهاتف ولا يضيّع المال.  \n\n3. عند الحنفية: يستمر في التعريف **سنةً كاملة** في اللقطة ذا

In [ ]:
import time
import random

curated_dataset = []
failed_records = []

# Configuration
MAX_RETRIES = 3
BASE_DELAY = 2 # Seconds to wait on success
ERROR_DELAY = 5 # Initial seconds to wait on error

print(f"Starting processing of {len(concatenated_list[336:])} items...")

for index, j in enumerate(concatenated_list[336:]):
    # Retry mechanism
    retries = 0
    success = False

    while retries < MAX_RETRIES and not success:
        try:
            print(f"Processing item {index + 1}/{len(concatenated_list[336:])}: {j["answer"][:30]}...")

            # --- Your Core Logic ---
            judge = judge_claude(j["answer"],j["llm_answer"] )
            # -----------------------

            # Append successful result
            curated_dataset.append({
                **j,
                "llm_judgement": judge,
                "llm_as_judge_modle":"claude-sonnet-4-5"
            })


            success = True

            # Rate Limit Protection: Sleep briefly on success to be polite to the API
            # Adding a small random jitter helps prevent synchronized API hits
            time.sleep(BASE_DELAY + random.uniform(0, 1))

        except Exception as e:
            retries += 1
            wait_time = ERROR_DELAY * retries # Linear backoff (5s, 10s, 15s)

            print(f"⚠️ Error on item {index} (Attempt {retries}/{MAX_RETRIES}): {e}")

            if retries < MAX_RETRIES:
                print(f"   Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed to process item {index} after {MAX_RETRIES} attempts.")
                # Log the failure so you don't lose track of it
                failed_records.append({"item": j, "error": str(e)})

# Summary
print("--- Processing Complete ---")
print(f"Successfully curated: {len(curated_dataset)}")
print(f"Failed items: {len(failed_records)}")

Streaming output truncated to the last 5000 lines.


I see the answers are fundamentally aligned. They converge on the core principle that trade intention determines zakat obligation. While Answer A explores the philosophical underpinnings, Answer B offers practical implementation details. Their perspectives are complementary, providing a holistic understanding of zakat requirements for traded land.

The nuanced elaboration in Answer B - explicitly mentioning nisab and hawl - doesn't challenge Answer A's reasoning but enriches it. These additional details are inherently consistent with the broader concept of زكاة عروض التجارة. I assess their alignment as precise, with Answer B serving as a more comprehensive exposition of the same underlying Islamic jurisprudential principle.

Response: 1
----------------------------------------------------------------------------------------------------
Processing item 150/264:     أقول وبالله التوفيق: لا تج...

Thinking summary: Let me analyze both a

In [ ]:
print(len(curated_dataset))

264


In [ ]:
import json

file_path = "/content/results_final_llm_answers_judge_2.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(curated_dataset, f, ensure_ascii=False, indent=2)

print(f"Curated dataset saved to {file_path}")

Curated dataset saved to /content/results_final_llm_answers_judge_2.json


In [ ]:
import json

file_path_1 = "/content/results_final_llm_answers_judge.json"
file_path_2 = "/content/results_final_llm_answers_judge_2.json"

# Load the first JSON file
with open(file_path_1, 'r', encoding='utf-8') as f:
    list_1 = json.load(f)

# Load the second JSON file
with open(file_path_2, 'r', encoding='utf-8') as f:
    list_2 = json.load(f)

# Concatenate the two lists
concatenated_list = list_1 + list_2

print(f"Length of the first list: {len(list_1)}")
print(f"Length of the second list: {len(list_2)}")
print(f"Length of the concatenated list: {len(concatenated_list)}")

Length of the first list: 336
Length of the second list: 264
Length of the concatenated list: 600


# Reading Data Judge

In [ ]:
import json

file_path_1 = "/content/results_final_llm_answers_judge_FINAL.json"

# Load the first JSON file
with open(file_path_1, 'r', encoding='utf-8') as f:
    concatenated_list = json.load(f)



print(f"Length of the first list: {len(concatenated_list)}")

Length of the first list: 600


In [ ]:
import json
from typing import List, Dict, Any

def calculate_llm_accuracy(data: List[Dict[str, Any]], target_llm: str) -> float:
    """
    Calculates the Simple Accuracy (Percentage of Correct Answers) for a specific LLM
    based on the 'llm_judgement' field.

    A score of '+1' (Identical meaning) is considered Correct.
    The 'llm_judgement' is stripped of non-digit/non-sign characters before comparison.

    Args:
        data: A list of dictionaries (the dataset).
        target_llm: The 'llm' name to filter and calculate accuracy for.

    Returns:
        The accuracy as a percentage (0.0 to 100.0).
    """
    # 1. Filter data for the target LLM
    llm_results = [item for item in data if item.get("llm") == target_llm]

    if not llm_results:
        print(f"No results found for LLM: {target_llm}")
        return 0.0

    total_results = len(llm_results)
    correct_count = 0

    # 2. Iterate and count correct answers ('+1' judgement)
    for result in llm_results:
        judgement_raw = result.get("llm_judgement")

        # Clean the judgement string: remove whitespace, newlines, and non-numeric characters
        # The 'strip' is crucial given the example: "llm_judgement": "**1**\n\nBoth answers convey..."
        if isinstance(judgement_raw, str):
            # Try to extract the score which might be inside non-digit characters
            score_text = judgement_raw.strip().replace('*', '').split('\n')[0].strip()

            # Simple check for the '1' score (which is the Correct one)
            if score_text == "1" or score_text == "+1":
                correct_count += 1

        # Fallback for the simpler cases shown in the sample
        elif judgement_raw == "1" or judgement_raw == 1:
            correct_count += 1

    # 3. Calculate Simple Accuracy
    accuracy = (correct_count / total_results) * 100.0

    print(f"--- Results for {target_llm} ---")
    print(f"Total evaluated results: {total_results}")
    print(f"Correct answers (+1 score): {correct_count}")
    print(f"Simple Accuracy: {accuracy:.2f}%")
    print("-" * (24 + len(target_llm)))

    return accuracy


# Calculate for gpt-5.1
gpt_accuracy = calculate_llm_accuracy(concatenated_list, "gpt-5.1")

# Calculate for gemini-2.5-flash
gemini_accuracy = calculate_llm_accuracy(concatenated_list, "gemini-2.5-flash")

--- Results for gpt-5.1 ---
Total evaluated results: 301
Correct answers (+1 score): 131
Simple Accuracy: 43.52%
-------------------------------
--- Results for gemini-2.5-flash ---
Total evaluated results: 299
Correct answers (+1 score): 139
Simple Accuracy: 46.49%
----------------------------------------


In [ ]:
import json

file_path = "/content/results_final_llm_answers_judge_FINAL.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(concatenated_list, f, ensure_ascii=False, indent=2)

print(f"Curated dataset saved to {file_path}")

Curated dataset saved to /content/results_final_llm_answers_judge_FINAL.json


In [ ]:
import json
from typing import List, Dict, Any
from collections import defaultdict

def calculate_accuracy_by_category(data: List[Dict[str, Any]], target_llm: str, category_key: str = "_fatherdepartment") -> Dict[str, float]:
    """
    Calculates the Accuracy per Category (percentage of Correct Answers) for a specific LLM.

    Args:
        data: A list of dictionaries (the dataset).
        target_llm: The 'llm' name to filter and calculate accuracy for.
        category_key: The dictionary key to group by (default: "_fatherdepartment").

    Returns:
        A dictionary where keys are categories and values are accuracy percentages.
    """

    # Structure: {'Category Name': {'total': 0, 'correct': 0}}
    category_stats = defaultdict(lambda: {'total': 0, 'correct': 0})

    # 1. Iterate through data
    filtered_count = 0
    for item in data:
        # Filter for the specific LLM
        if item.get("llm") != target_llm:
            continue

        filtered_count += 1

        # Get the category (handle missing keys)
        category = item.get(category_key, "Unknown Category")

        # Check judgement logic
        judgement_raw = item.get("llm_judgement")
        is_correct = False

        # Clean the judgement string (Same logic as your original script)
        if isinstance(judgement_raw, str):
            # Remove asterisks, whitespace, take first line
            score_text = judgement_raw.strip().replace('*', '').split('\n')[0].strip()
            if score_text == "1" or score_text == "+1":
                is_correct = True
        elif judgement_raw == 1 or judgement_raw == "1":
            is_correct = True

        # Update stats for this specific category
        category_stats[category]['total'] += 1
        if is_correct:
            category_stats[category]['correct'] += 1

    if filtered_count == 0:
        print(f"No results found for LLM: {target_llm}")
        return {}

    # 2. Calculate and Print Results
    final_results = {}

    print(f"\n=== Accuracy Breakdown for: {target_llm} ===")
    print(f"{'Category':<30} | {'Accuracy':<10} | {'Count (Correct/Total)'}")
    print("-" * 65)

    for category, stats in category_stats.items():
        total = stats['total']
        correct = stats['correct']

        if total > 0:
            accuracy = (correct / total) * 100.0
        else:
            accuracy = 0.0

        final_results[category] = accuracy
        print(f"{category:<30} | {accuracy:>6.2f}%   | {correct}/{total}")

    print("-" * 65)
    return final_results



# --- Execution ---

# 1. Calculate for gpt-5.1
# Note: Based on the sample data above:
# 'المعاملات والأمانات' has 2 items: one -1, one 1. (50% Accuracy)
# 'العبادات' has 1 item: one 1. (100% Accuracy)
gpt_stats = calculate_accuracy_by_category(concatenated_list, "gpt-5.1")

# 2. Calculate for gemini-2.5-flash
gemini_stats = calculate_accuracy_by_category(concatenated_list, "gemini-2.5-flash")


=== Accuracy Breakdown for: gpt-5.1 ===
Category                       | Accuracy   | Count (Correct/Total)
-----------------------------------------------------------------
المعاملات والأمانات            |  41.67%   | 25/60
الحظر والإباحة واللباس والزينة |  46.67%   | 28/60
الذكر والتزكية والأخلاق        |  39.34%   | 24/61
الزواج والطلاق والنفقات        |  55.00%   | 33/60
العبادات                       |  35.00%   | 21/60
-----------------------------------------------------------------

=== Accuracy Breakdown for: gemini-2.5-flash ===
Category                       | Accuracy   | Count (Correct/Total)
-----------------------------------------------------------------
المعاملات والأمانات            |  42.37%   | 25/59
الحظر والإباحة واللباس والزينة |  43.33%   | 26/60
الذكر والتزكية والأخلاق        |  51.67%   | 31/60
الزواج والطلاق والنفقات        |  51.67%   | 31/60
العبادات                       |  43.33%   | 26/60
-----------------------------------------------------------------